
# 03. 오디오비주얼로 — 추출 정보로 웹 오디오비주얼 만들기

이 노트북은 **자기 피아노 연주**에서 AI가 필요한 정보를 추출하고, 그 결과를 **브라우저 기반 오디오비주얼**로 연결하는 데모입니다.
수업에서는 먼저 준비된 예시 피아노 음원으로 시연하고, 이후 자신의 연주로 바꿔보는 흐름을 상정합니다.

**파이프라인**
- 입력: 피아노 연주 오디오
- AI 추출: `Basic Pitch`로 MIDI / note event 만들기
- cue 생성: energy, density, register, beat 같은 제어 정보 만들기
- 출력: `p5.js WEBGL` 기반 브라우저 오디오비주얼

▶ Colab에서 실행한다면 아래 설치 셀부터 시작하세요.
서버 시연 환경에서 이미 `setup.sh`를 실행했다면, 설치 셀은 건너뛰고 다음 셀로 바로 넘어가도 됩니다.


In [ ]:

# Colab용 설치 (서버 시연 환경에서는 보통 건너뛰어도 됩니다)
!pip install -q basic-pitch librosa soundfile pretty_midi numpy matplotlib

import warnings
warnings.filterwarnings('ignore')

print('✅ 설치 완료!')



---
## 1. 준비된 피아노 예시 선택

수업에서는 먼저 **이미 준비된 피아노 음원**으로 데모를 진행합니다.
아래 셀은 먼저 repo의 `assets/` 폴더를 확인하고, 없으면 예시 파일을 다운로드합니다.


In [ ]:

import os
import urllib.request
from pathlib import Path
import IPython.display as ipd

REPO_URL = 'https://raw.githubusercontent.com/joonhyungbae/pianokit/main/assets'
ASSETS_DIR = Path('assets')

examples = {
    'piano_chopin.wav': '쇼팽 스타일 피아노 (서정적, 데모용 기본값)',
    'piano_jazz.wav': '재즈 피아노 즉흥',
    'piano_simple.wav': '짧고 단순한 피아노 프레이즈',
}

print('🎹 사용 가능한 준비된 피아노 예시')
resolved = {}
for fname, desc in examples.items():
    local_asset = ASSETS_DIR / fname
    local_copy = Path(fname)
    if local_asset.exists():
        resolved[fname] = str(local_asset)
        print(f'  ✅ {fname}: assets 폴더에서 사용')
    elif local_copy.exists():
        resolved[fname] = str(local_copy)
        print(f'  ✅ {fname}: 현재 디렉터리에서 사용')
    else:
        try:
            urllib.request.urlretrieve(f'{REPO_URL}/{fname}', fname)
            resolved[fname] = fname
            print(f'  ✅ {fname}: 다운로드 완료')
        except Exception:
            print(f'  ⚠️ {fname}: 준비 실패')
            continue
    print(f'  - {desc}')

input_audio = resolved.get('piano_chopin.wav')

if input_audio and os.path.exists(input_audio):
    print()
    print(f'🎵 현재 선택된 피아노 음원: {input_audio}')
    ipd.display(ipd.Audio(input_audio))
else:
    raise FileNotFoundError('예시 피아노 음원을 찾지 못했습니다. assets 폴더를 확인해주세요.')



---
## 2. AI로 표기 정보 추출하기

이 단계에서는 `Basic Pitch`를 이용해 오디오에서 **MIDI와 note event**를 추출합니다.
이 정보는 나중에 화면을 제어하는 cue의 바탕이 됩니다.


In [ ]:

import json
import numpy as np
import librosa
import pretty_midi
from pathlib import Path
from basic_pitch.inference import predict

print('🔄 Basic Pitch로 표기 정보를 추출하는 중...')
model_output, midi_data, note_events = predict(input_audio)

output_midi = 'visual_output.mid'
midi_data.write(output_midi)
pm = pretty_midi.PrettyMIDI(output_midi)

notes = []
for instrument in pm.instruments:
    for note in instrument.notes:
        notes.append({
            'start': float(note.start),
            'end': float(note.end),
            'pitch': int(note.pitch),
            'velocity': int(note.velocity),
        })

notes.sort(key=lambda x: x['start'])

y, sr = librosa.load(input_audio, sr=22050)
duration = len(y) / sr

tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
tempo = float(tempo) if hasattr(tempo, '__len__') else float(tempo)
beat_times = librosa.frames_to_time(beats, sr=sr).tolist()

rms = librosa.feature.rms(y=y)[0]
rms_times = librosa.times_like(rms, sr=sr)
centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
centroid_times = librosa.times_like(centroid, sr=sr)

print(f'✅ MIDI 생성 완료: {output_midi}')
print(f'🎼 추출된 note event 수: {len(notes)}')
print(f'🥁 추정 템포: {tempo:.0f} BPM')



---
## 3. 오디오비주얼 cue 만들기

이제 MIDI와 오디오 특성을 이용해 브라우저 장면을 제어할 cue를 만듭니다.
여기서는 다음 정보를 사용합니다.

- `energy`: 음량 변화
- `density`: 동시에 울리는 음의 밀도
- `register`: 저음/중음/고음의 중심
- `brightness`: 음색의 밝기
- `attack`: 새 음의 시작 강도
- `beat`: 박자 이벤트


In [ ]:

import matplotlib.pyplot as plt

fps = 24
frame_times = np.arange(0, duration, 1 / fps)
energy = np.interp(frame_times, rms_times, rms)
brightness = np.interp(frame_times, centroid_times, centroid)

density = np.zeros_like(frame_times)
register_sum = np.zeros_like(frame_times)
active_counts = np.zeros_like(frame_times)
attack = np.zeros_like(frame_times)

for note in notes:
    start_idx = int(np.clip(np.floor(note['start'] * fps), 0, len(frame_times) - 1))
    end_idx = int(np.clip(np.ceil(note['end'] * fps), 0, len(frame_times) - 1))

    density[start_idx:end_idx + 1] += 1
    register_sum[start_idx:end_idx + 1] += note['pitch']
    active_counts[start_idx:end_idx + 1] += 1
    attack[start_idx] += 1

register = np.where(active_counts > 0, register_sum / np.maximum(active_counts, 1), 60)


def normalize(arr):
    arr = np.asarray(arr, dtype=float)
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

energy_norm = normalize(energy)
density_norm = normalize(density)
register_norm = normalize(register)
brightness_norm = normalize(brightness)
attack_norm = normalize(attack)

beat_flags = np.zeros_like(frame_times)
for bt in beat_times:
    idx = int(np.clip(round(bt * fps), 0, len(frame_times) - 1))
    beat_flags[idx] = 1.0

visual_frames = []
for i, t in enumerate(frame_times):
    visual_frames.append({
        't': round(float(t), 4),
        'energy': round(float(energy_norm[i]), 4),
        'density': round(float(density_norm[i]), 4),
        'register': round(float(register_norm[i]), 4),
        'brightness': round(float(brightness_norm[i]), 4),
        'attack': round(float(attack_norm[i]), 4),
        'beat': int(beat_flags[i]),
    })

visual_cues = {
    'audio_file': input_audio,
    'midi_file': output_midi,
    'duration': round(float(duration), 3),
    'tempo': round(float(tempo), 2),
    'fps': fps,
    'frames': visual_frames,
}

with open('visual_cues.json', 'w', encoding='utf-8') as f:
    json.dump(visual_cues, f, ensure_ascii=False, indent=2)

print('✅ cue 저장 완료: visual_cues.json')
print(f'🎞️ 총 프레임 수: {len(visual_frames)}')

fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
axes[0].plot(frame_times, energy_norm, color='#4f46e5')
axes[0].set_ylabel('energy')
axes[0].set_title('오디오비주얼 cue 미리보기')

axes[1].plot(frame_times, density_norm, color='#0f766e')
axes[1].set_ylabel('density')

axes[2].plot(frame_times, register_norm, color='#ca8a04')
axes[2].set_ylabel('register')

axes[3].plot(frame_times, brightness_norm, color='#be185d', alpha=0.8)
axes[3].scatter(frame_times[beat_flags > 0], np.ones(int(beat_flags.sum())), color='#ef4444', s=12, label='beat')
axes[3].set_ylabel('brightness')
axes[3].set_xlabel('time (s)')
axes[3].legend(loc='upper right')

plt.tight_layout()
plt.show()



---
## 4. p5.js WEBGL로 오디오비주얼 보기

아래 셀은 `visual_cues.json`과 오디오를 브라우저에 넘겨 **p5.js WEBGL** 장면을 만듭니다.
재생 버튼을 누르면 음악과 함께 장면이 반응합니다.

- `energy`는 중심 오브젝트의 규모와 밝기를 바꿉니다.
- `density`는 주변 파티클과 격자 밀도를 바꿉니다.
- `register`는 색과 카메라 각도를 움직입니다.
- `beat`와 `attack`은 순간적인 pulse를 만듭니다.


In [ ]:

import base64
from uuid import uuid4
from IPython.display import HTML, display

container_id = f"av-{uuid4().hex}"
audio_bytes = Path(input_audio).read_bytes()
audio_b64 = base64.b64encode(audio_bytes).decode('ascii')
cues_json = json.dumps(visual_cues, ensure_ascii=False)

audio_mime = 'audio/wav' if input_audio.lower().endswith('.wav') else 'audio/mpeg'

html = f"""
<div id="{container_id}" style="font-family: 'IBM Plex Sans KR', sans-serif; color: #e5e7eb; background: linear-gradient(135deg, #050816, #101828); border: 1px solid rgba(255,255,255,0.08); border-radius: 24px; padding: 20px;">
  <div style="display:flex; justify-content:space-between; align-items:center; gap:16px; flex-wrap:wrap; margin-bottom:14px;">
    <div>
      <div style="font-size:28px; font-weight:800;">Cue-driven Browser Audiovisual</div>
      <div style="font-size:15px; color:#94a3b8; margin-top:4px;">Basic Pitch로 추출한 정보가 p5.js WEBGL 장면을 제어합니다.</div>
    </div>
    <button id="{container_id}-toggle" style="border:none; border-radius:999px; padding:12px 18px; background:#f59e0b; color:#111827; font-size:15px; font-weight:700; cursor:pointer;">재생 시작</button>
  </div>
  <div style="display:flex; gap:10px; flex-wrap:wrap; margin-bottom:14px; color:#cbd5e1; font-size:13px;">
    <span style="padding:6px 10px; border:1px solid rgba(255,255,255,0.12); border-radius:999px;">tempo: {visual_cues['tempo']} BPM</span>
    <span style="padding:6px 10px; border:1px solid rgba(255,255,255,0.12); border-radius:999px;">fps: {visual_cues['fps']}</span>
    <span style="padding:6px 10px; border:1px solid rgba(255,255,255,0.12); border-radius:999px;">renderer: p5.js WEBGL</span>
  </div>
  <div id="{container_id}-canvas" style="min-height:540px; border-radius:20px; overflow:hidden; background:#020617;"></div>
  <div id="{container_id}-readout" style="display:grid; grid-template-columns:repeat(5, minmax(0, 1fr)); gap:10px; margin-top:14px; font-size:13px; color:#cbd5e1;"></div>
</div>
<script src="https://cdn.jsdelivr.net/npm/p5@1.11.1/lib/p5.min.js"></script>
<script>
(() => {{
  const data = {cues_json};
  const frames = data.frames;
  const audio = new Audio('data:{audio_mime};base64,{audio_b64}');
  const toggle = document.getElementById('{container_id}-toggle');
  const readout = document.getElementById('{container_id}-readout');
  const host = document.getElementById('{container_id}-canvas');
  let isReady = false;

  function frameAt(time) {{
    if (!frames.length) return null;
    const idx = Math.min(Math.floor(time * data.fps), frames.length - 1);
    return frames[idx];
  }}

  function renderReadout(frame) {{
    if (!frame) return;
    const entries = [
      ['time', frame.t.toFixed(2) + 's'],
      ['energy', frame.energy.toFixed(2)],
      ['density', frame.density.toFixed(2)],
      ['register', frame.register.toFixed(2)],
      ['brightness', frame.brightness.toFixed(2)],
    ];
    readout.innerHTML = entries.map(([k, v]) => `
      <div style="border:1px solid rgba(255,255,255,0.10); border-radius:16px; padding:10px 12px; background:rgba(15,23,42,0.55);">
        <div style="font-size:11px; text-transform:uppercase; letter-spacing:0.12em; color:#f59e0b;">${{k}}</div>
        <div style="font-size:18px; font-weight:700; margin-top:4px; color:#f8fafc;">${{v}}</div>
      </div>
    `).join('');
  }}

  toggle.addEventListener('click', async () => {{
    if (audio.paused) {{
      await audio.play();
      toggle.textContent = '일시정지';
    }} else {{
      audio.pause();
      toggle.textContent = '재생 시작';
    }}
  }});

  audio.addEventListener('ended', () => {{
    toggle.textContent = '다시 재생';
  }});

  new p5((p) => {{
    const particleCount = 96;
    const particles = Array.from({{ length: particleCount }}, (_, i) => ({{
      angle: (Math.PI * 2 * i) / particleCount,
      depth: p.random(-220, 220),
      seed: p.random(1000),
    }}));

    p.setup = () => {{
      const canvas = p.createCanvas(host.clientWidth || 900, 540, p.WEBGL);
      canvas.parent(host);
      p.colorMode(p.HSB, 360, 100, 100, 100);
      p.noStroke();
      isReady = true;
      renderReadout(frameAt(0));
    }};

    p.windowResized = () => {{
      p.resizeCanvas(host.clientWidth || 900, 540);
    }};

    p.draw = () => {{
      const now = audio.currentTime || 0;
      const frame = frameAt(now) || frames[0];
      if (!frame) return;
      renderReadout(frame);

      const energy = frame.energy;
      const density = frame.density;
      const reg = frame.register;
      const bright = frame.brightness;
      const attack = frame.attack;
      const beat = frame.beat;

      const hue = 210 + reg * 90;
      const pulse = 1 + energy * 0.6 + beat * 0.35;
      const orbit = now * 0.45 + reg * 1.8;

      p.background(228, 60, 7 + bright * 8);
      p.ambientLight(220, 20, 35);
      p.pointLight(hue, 70, 100, 0, -180, 260);
      p.pointLight((hue + 140) % 360, 60, 100, 180, 120, 120);

      p.push();
      p.rotateX(-0.25 + reg * 0.35);
      p.rotateY(orbit * 0.6);
      p.fill(hue, 55, 24 + bright * 20, 22);
      p.plane(900, 540);
      p.pop();

      p.push();
      p.rotateY(orbit);
      p.rotateX(0.75 + reg * 0.2);
      for (let i = 0; i < 34; i++) {{
        const x = p.map(i, 0, 33, -260, 260);
        const h = 35 + energy * 150 + density * 80 * Math.sin(now * 1.2 + i * 0.3);
        const w = 10 + density * 10;
        p.push();
        p.translate(x, -20 + Math.sin(now * 2.0 + i * 0.25) * 12, -120 + i * 6);
        p.fill((hue + i * 3.5) % 360, 60, 95, 70);
        p.box(w, Math.abs(h), 12 + bright * 18);
        p.pop();
      }}
      p.pop();

      p.push();
      p.rotateY(-orbit * 0.7);
      p.rotateZ(now * 0.25);
      p.fill((hue + 40) % 360, 40, 100, 18 + energy * 14);
      p.sphere(90 * pulse, 48, 32);
      p.pop();

      p.push();
      p.blendMode(p.ADD);
      particles.forEach((particle, idx) => {{
        const radius = 150 + density * 140 + 20 * Math.sin(now * 0.9 + particle.seed);
        const theta = particle.angle + orbit * 0.8 + idx * 0.003;
        const x = Math.cos(theta) * radius;
        const y = Math.sin(theta * 1.8 + particle.seed) * (70 + energy * 90);
        const z = particle.depth + Math.sin(now + particle.seed) * 40;
        const size = 2 + bright * 7 + attack * 18;
        p.push();
        p.translate(x, y, z);
        p.fill((hue + 120 + idx) % 360, 35, 100, 55);
        p.sphere(size, 8, 8);
        p.pop();
      }});
      p.blendMode(p.BLEND);
      p.pop();

      if (beat > 0) {{
        p.push();
        p.rotateX(Math.PI / 2);
        p.noFill();
        p.stroke((hue + 180) % 360, 50, 100, 65);
        p.strokeWeight(3);
        p.torus(170 + energy * 60, 7 + attack * 12, 40, 10);
        p.noStroke();
        p.pop();
      }}
    }};
  }});
}})();
</script>
"""

display(HTML(html))



---
## 5. 결과 저장

아래 셀은 이번 단계에서 만들어진 핵심 결과 파일을 확인합니다.

- `visual_output.mid`: 오디오에서 추출한 MIDI
- `visual_cues.json`: 브라우저 장면 제어용 cue 데이터


In [ ]:

from pathlib import Path

for filename in ['visual_output.mid', 'visual_cues.json']:
    path = Path(filename)
    if path.exists():
        print(f'✅ {filename}: {path.resolve()}')
    else:
        print(f'⚠️ {filename}: 생성되지 않았습니다.')

print('\nℹ️ 이 cue 파일은 이후 웹 앱이나 p5.js 프로토타입에서 그대로 다시 쓸 수 있습니다.')
